In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import torch
from tqdm import tqdm

def evaluate_rouge(test_dataset, model_name="/home/jovyan/shares/SR004.nfs2/amaksimova/models/cultura_1.3b_summarize", batch_size=4, max_length=128):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    
    # Загрузка метрики
    rouge = evaluate.load("rouge")
    
    # Подготовка данных
    def generate_summaries(texts):
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True
        )
        return tokenizer.batch_decode(outputs, skip_special_tokens=True)
    
    # Вычисление метрик
    predictions = []
    references = []
    
    for i in tqdm(range(0, len(test_dataset), batch_size)):
        batch = test_dataset[i:i+batch_size]
        batch_texts = batch["text"]
        batch_summaries = batch["summary"]
        
        preds = generate_summaries(batch_texts)
        predictions.extend(preds)
        references.extend(batch_summaries)
    
    metrics = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    return {k: round(v, 4) for k, v in metrics.items()}

from datasets import load_dataset
test_data = load_dataset("RussianNLP/Mixed-Summarization-Dataset",split="test[:100]")
metrics = evaluate_rouge(test_data)
print("ROUGE Metrics:", metrics)

100%|██████████| 25/25 [01:37<00:00,  3.89s/it]

ROUGE Metrics: {'rouge1': 0.2214, 'rouge2': 0.1224, 'rougeL': 0.2166, 'rougeLsum': 0.218}
